# Step 7b – MOS listening test: ফলাফল বিশ্লেষণ
Google Form-এর উত্তর ("MOS responses" Sheet) আর `mos_key.csv` মিলিয়ে প্রতিটা সিস্টেমের নম্বর বের করে।

- **GPU লাগে না** (CPU runtime-ই যথেষ্ট)। **Run all**।
- প্রথম cell-এ Google অ্যাকাউন্টে অনুমতি চাইবে (Sheet পড়ার জন্য) — নিজের অ্যাকাউন্ট বেছে Allow দিন। কোনো password বা token লাগবে না।
- Sheet খুঁজে না পেলে: Sheet খুলে **File → Download → CSV** করে Drive-এ `voice_clone/mos_test/MOS responses.csv` নামে রাখুন, notebook নিজে সেটা পড়বে।
- ফল সেভ হয় `voice_clone/mos_test/results/`-এ। কিছু মোছে না, Sheet-এ কিছু লেখে না।

**হিসাব:** প্রতিটা সিস্টেমের গড় নম্বর (MOS) আর 95% confidence interval (শ্রোতা ধরে bootstrap, 10,000 বার, seed 13)। দুই সিস্টেমের তুলনা: প্রতিটা শ্রোতার গড় নিয়ে Wilcoxon signed-rank test, Holm correction।

## 1. Drive, অনুমতি আর ফাইল

In [ ]:
import os, re, numpy as np, pandas as pd
from google.colab import drive, auth
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/Sylheti_Thesis/voice_clone/mos_test"
KEY = BASE + "/answer_key_do_not_share/mos_key.csv"
OUT = BASE + "/results"; os.makedirs(OUT, exist_ok=True)
assert os.path.exists(KEY), "mos_key.csv পাওয়া যায়নি: " + KEY
key = pd.read_csv(KEY, encoding="utf-8-sig")

CSV = BASE + "/MOS responses.csv"
if os.path.exists(CSV):
    resp = pd.read_csv(CSV)
    print("read CSV:", CSV)
else:
    auth.authenticate_user()                 # Google login popup (no password is stored in the notebook)
    import gspread, google.auth
    creds, _ = google.auth.default()
    gc = gspread.authorize(creds)
    ws = gc.open("MOS responses").sheet1
    resp = pd.DataFrame(ws.get_all_records())
    print("read Google Sheet: MOS responses")
print(len(resp), "listeners |", len(resp.columns), "columns")

## 2. উত্তরগুলো সাজানো (প্রতিটা নম্বর এক লাইনে)

In [ ]:
SYS_NAME = {"real": "Real recording", "mms_base": "MMS-TTS", "indic_parler": "Indic Parler-TTS", "v1_clone": "MMS-TTS adapted to V1"}
lang_col = next((c for c in resp.columns if "মাতৃভাষা" in c), None)
head_col = next((c for c in resp.columns if "হেডফোন" in c), None)
resp = resp.reset_index(drop=True)
resp["listener"] = ["L%02d" % (i + 1) for i in range(len(resp))]

def score(v):
    m = re.match(r"\s*([1-5])", str(v))
    return int(m.group(1)) if m else np.nan

rows = []
for c in resp.columns:
    m = re.match(r"^অডিও (\d+) \[(.+)\]$", str(c))
    if m:
        f = "audio_%02d.wav" % int(m.group(1))
        q = "naturalness" if m.group(2).startswith("স্বাভাবিকতা") else "intelligibility"
        for _, r in resp.iterrows():
            rows.append({"listener": r.listener, "file": f, "question": q, "score": score(r[c])})
        continue
    m = re.match(r"^জোড়া (\d+):", str(c))
    if m:
        f = "sim_%02d.wav" % int(m.group(1))
        for _, r in resp.iterrows():
            rows.append({"listener": r.listener, "file": f, "question": "similarity", "score": score(r[c])})
long = pd.DataFrame(rows).merge(key[["file", "system", "sent_id"]], on="file", how="left")
assert long.system.notna().all(), "কিছু ফাইল mos_key.csv-তে নেই"
long = long.dropna(subset=["score"])
if lang_col: long = long.merge(resp[["listener", lang_col]].rename(columns={lang_col: "native"}), on="listener")
if head_col: long = long.merge(resp[["listener", head_col]].rename(columns={head_col: "headphones"}), on="listener")
long.to_csv(OUT + "/mos_long.csv", index=False, encoding="utf-8-sig")
print(long.groupby(["question", "system"]).size().unstack(0))
if lang_col: print("\nশ্রোতার মাতৃভাষা:", resp[lang_col].value_counts().to_dict())
if head_col: print("হেডফোন:", resp[head_col].value_counts().to_dict())

## 3. MOS আর 95% CI

In [ ]:
N_BOOT, SEED = 10_000, 13
def mos_ci(d):
    """mean score and 95% CI, resampling listeners (each listener's ratings stay together)"""
    lis = d.listener.unique()
    per = {l: g.score.values for l, g in d.groupby("listener")}
    rng = np.random.default_rng(SEED)
    boots = []
    for _ in range(N_BOOT):
        pick = rng.choice(lis, size=len(lis), replace=True)
        boots.append(np.concatenate([per[l] for l in pick]).mean())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return round(d.score.mean(), 2), round(lo, 2), round(hi, 2), d.listener.nunique(), len(d)

ORDER = ["real", "mms_base", "indic_parler", "v1_clone"]
res = []
for q in ["naturalness", "intelligibility", "similarity"]:
    for s in ORDER:
        d = long[(long.question == q) & (long.system == s)]
        if len(d):
            m, lo, hi, nl, nr = mos_ci(d)
            res.append({"question": q, "system": SYS_NAME[s], "MOS": m, "95% CI": f"[{lo:.2f}, {hi:.2f}]",
                        "listeners": nl, "ratings": nr})
res = pd.DataFrame(res)
res.to_csv(OUT + "/mos_results.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.width", 200)
print(res.to_string(index=False))
print("\nsaved:", OUT + "/mos_results.csv")

## 4. দুই সিস্টেমের তুলনা (Wilcoxon signed-rank, Holm correction)

In [ ]:
from itertools import combinations
from scipy.stats import wilcoxon
tests = []
for q in ["naturalness", "intelligibility"]:
    per = long[long.question == q].groupby(["listener", "system"]).score.mean().unstack()
    pairs = [(a, b) for a, b in combinations(ORDER, 2) if a in per and b in per]
    ps = []
    for a, b in pairs:
        x = per[[a, b]].dropna()
        try:
            p = wilcoxon(x[a], x[b]).pvalue
        except ValueError:          # all differences zero
            p = 1.0
        ps.append(p)
        tests.append({"question": q, "A": SYS_NAME[a], "B": SYS_NAME[b], "listeners": len(x),
                      "mean A": round(x[a].mean(), 2), "mean B": round(x[b].mean(), 2), "p": p})
    # Holm correction within each question
    idx = [i for i, t in enumerate(tests) if t["question"] == q]
    order = sorted(idx, key=lambda i: tests[i]["p"]); m = len(order); run = 0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * tests[i]["p"]))
        tests[i]["p (Holm)"] = run
tests = pd.DataFrame(tests)
tests["p"] = tests["p"].round(4); tests["p (Holm)"] = tests["p (Holm)"].round(4)
tests["significant"] = np.where(tests["p (Holm)"] < 0.05, "yes", "no")
tests.to_csv(OUT + "/mos_pairwise.csv", index=False, encoding="utf-8-sig")
print(tests.to_string(index=False))
print("\nsaved:", OUT + "/mos_pairwise.csv")

## 5. শ্রোতার মাতৃভাষা অনুযায়ী (অতিরিক্ত) আর ছবি (thesis figure)

In [ ]:
import matplotlib.pyplot as plt
if "native" in long:
    t = (long[long.question != "similarity"].groupby(["question", "native", "system"]).score.mean()
         .unstack("system")[[s for s in ORDER if s in long.system.unique()]].round(2))
    t.columns = [SYS_NAME[c] for c in t.columns]
    print(t.to_string())
    t.to_csv(OUT + "/mos_by_native_language.csv", encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, q, title in zip(axes, ["naturalness", "intelligibility"], ["(a) Naturalness", "(b) Intelligibility"]):
    r = res[res.question == q]
    lo = [float(c.strip("[]").split(",")[0]) for c in r["95% CI"]]
    hi = [float(c.strip("[]").split(",")[1]) for c in r["95% CI"]]
    y = r.MOS.values
    ax.bar(range(len(r)), y, color="#4C78A8", width=0.6)
    ax.errorbar(range(len(r)), y, yerr=[y - np.array(lo), np.array(hi) - y], fmt="none", ecolor="black", capsize=4)
    ax.set_xticks(range(len(r))); ax.set_xticklabels(r.system, rotation=20, ha="right", fontsize=9)
    ax.set_ylim(1, 5); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Mean opinion score (1–5)")
plt.tight_layout(); plt.savefig(OUT + "/Figure_MOS.png", dpi=200); plt.show()
print("saved:", OUT + "/Figure_MOS.png")

## 6. Drive-এ সব লিখে unmount

In [ ]:
drive.flush_and_unmount()
print("done. ফলাফল:", OUT)